In [ ]:
%%sql -r dataframe_1
USE DATABASE LUMISENSE_DB;

USE SCHEMA LUMISENSE_DB.LUMISENSE_SCHEMA;

In [ ]:
# ═══════════════════════════════════════════════════════════════
# LUMISENSE GRID — FACT TABLE POPULATION
# ═══════════════════════════════════════════════════════════════

import random
import uuid
from datetime import datetime, timedelta

# Add this at the very top of your cell — before everything else

from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
# ─── ZONES ────────────────────────────────────────────────────
ZONES = {
    "Residential": { "lights": ["L1","L2","L3","L4","L5"],    "type": "Sub-Urban",  "ldr_offset": 0   },
    "Main Road":   { "lights": ["L6","L7","L8","L9","L10"],   "type": "Urban",      "ldr_offset": 50  },
    "Park":        { "lights": ["L11","L12","L13","L14","L15"],"type": "Green",      "ldr_offset": 100 },
    "Industrial":  { "lights": ["L16","L17","L18","L19","L20"],"type": "Commercial", "ldr_offset": -80 },
}

LIGHT_VARIATION = {f"L{i}": random.randint(-25, 25) for i in range(1, 21)}

# ─── RULES (saving = 100 - brightness) ───────────────────────
RULES = [
    (1,  0,   299,  0,  5,  50,  'ECO',    50),
    (2,  0,   299,  6,  8,  95,  'FULL',   5),
    (3,  0,   299,  9,  17, 80,  'NORMAL', 20),
    (4,  0,   299,  18, 21, 100, 'FULL',   0),
    (5,  0,   299,  22, 23, 60,  'NORMAL', 40),
    (6,  300, 599,  0,  5,  20,  'ECO',    80),
    (7,  300, 599,  6,  8,  60,  'NORMAL', 40),
    (8,  300, 599,  9,  17, 30,  'ECO',    70),
    (9,  300, 599,  18, 21, 80,  'FULL',   20),
    (10, 300, 599,  22, 23, 35,  'NORMAL', 65),
    (11, 600, 1023, 0,  5,  10,  'ECO',    90),
    (12, 600, 1023, 6,  8,  30,  'ECO',    70),
    (13, 600, 1023, 9,  17, 5,   'ECO',    95),
    (14, 600, 1023, 18, 21, 50,  'NORMAL', 50),
    (15, 600, 1023, 22, 23, 15,  'ECO',    85),
]

SCENARIO_LABELS = {
    1:  "Dark Night — ECO",          2:  "Dark Dawn — Full Power",
    3:  "Dark Daytime — Normal",     4:  "Dark Evening — Full Power",
    5:  "Dark Late Night — Normal",  6:  "Moderate Night — ECO",
    7:  "Moderate Morning — Normal", 8:  "Moderate Daytime — ECO",
    9:  "Moderate Evening — Full",   10: "Moderate Late Night — Normal",
    11: "Bright Night — Deep ECO",   12: "Bright Morning — ECO",
    13: "Bright Daytime — Deep ECO", 14: "Bright Evening — Normal",
    15: "Bright Late Night — ECO",
}


In [ ]:

# ─── HELPERS ──────────────────────────────────────────────────
def get_ldr(hour, zone):
    offset = ZONES[zone]["ldr_offset"]
    if   0 <= hour <= 5:  base = random.randint(30,  150)
    elif 6 <= hour <= 8:  base = random.randint(200, 500)
    elif 9 <= hour <= 17: base = random.randint(600, 950)
    elif 18 <= hour <= 21:base = random.randint(250, 550)
    else:                 base = random.randint(80,  250)
    return max(0, min(1023, base + offset + random.randint(-25, 25)))

def get_ldr_category(ldr):
    if ldr < 300:   return "Dark"
    elif ldr < 600: return "Moderate"
    else:           return "Bright"

def get_time_period(hour):
    if   0 <= hour <= 5:  return "Midnight"
    elif 6 <= hour <= 8:  return "Morning"
    elif 9 <= hour <= 17: return "Daytime"
    elif 18 <= hour <= 21:return "Evening"
    else:                  return "Late Night"

def get_rule(ldr, hour):
    for r in RULES:
        if r[1] <= ldr <= r[2] and r[3] <= hour <= r[4]:
            return r[0], r[5], r[6], r[7]
    return 1, 50, 'NORMAL', 50

def build_row(dt, light_id, zone):
    hour        = dt.hour
    ldr         = get_ldr(hour, zone)
    ldr         = max(0, min(1023, ldr + LIGHT_VARIATION.get(light_id, 0)))
    scenario_id, brightness, mode, _ = get_rule(ldr, hour)
    saving      = 100 - brightness          # always 100 - brightness

    wattage     = 150.0
    actual_w    = round(wattage * brightness / 100, 2)
    power_saved = round(wattage - actual_w, 2)
    cost_kwh    = 0.12
    interval_hrs= 0.25
    cost_saved  = round(power_saved * interval_hrs / 1000 * cost_kwh, 5)
    co2_saved   = round(power_saved * interval_hrs / 1000 * 0.233, 5)
    voltage     = round(random.uniform(228, 238), 1)
    current     = round(actual_w / voltage, 3)
    grid_load   = round((actual_w / wattage) * 100, 1)
    day_name    = dt.strftime("%A")
    is_weekend  = day_name in ("Saturday", "Sunday")
    month_name  = dt.strftime("%B")

    return (
        str(uuid.uuid4()),
        dt.strftime('%Y-%m-%d %H:%M:%S'),
        light_id,
        zone,
        ZONES[zone]["type"],
        ldr,
        get_ldr_category(ldr),
        hour,
        get_time_period(hour),
        day_name,
        is_weekend,
        month_name,
        brightness,
        mode,
        saving,
        wattage,
        actual_w,
        power_saved,
        cost_kwh,
        cost_saved,
        co2_saved,
        voltage,
        current,
        grid_load,
        scenario_id,
        SCENARIO_LABELS[scenario_id],
    )

# ─── STEP 1: CLEAR EXISTING DATA ──────────────────────────────
print("🗑️  Clearing existing fact_dashboard data...")
session.sql("DELETE FROM LUMISENSE_DB.LUMISENSE_SCHEMA.fact_dashboard").collect()
print("✅ Cleared!\n")


In [ ]:

# ─── STEP 2: GENERATE ROWS ────────────────────────────────────
print("🏗️  Generating rows — 30 days × 96 intervals × 20 lights...")

now          = datetime.now().replace(second=0, microsecond=0)
now          = now - timedelta(minutes=now.minute % 15)
start        = now - timedelta(days=30)
current_time = start
rows         = []

while current_time <= now:
    for zone, data in ZONES.items():
        for light_id in data["lights"]:
            rows.append(build_row(current_time, light_id, zone))
    current_time += timedelta(minutes=15)

print(f"✅ {len(rows):,} rows generated!\n")

# ─── STEP 3: BATCH INSERT ─────────────────────────────────────
print("📦 Inserting in batches of 2,000...\n")

insert_sql = """
    INSERT INTO LUMISENSE_DB.LUMISENSE_SCHEMA.fact_dashboard (
        fact_id, snapshot_time, light_id, zone, zone_type,
        ldr_value, ldr_category, hour_of_day, time_period,
        day_of_week, is_weekend, month_name,
        brightness_pct, energy_mode, saving_pct,
        wattage_per_light, actual_wattage, power_saved_w,
        cost_per_kwh, cost_saved_usd, co2_saved_kg,
        voltage_v, current_a, grid_load_pct,
        scenario_id, scenario_label
    ) VALUES (
        ?, ?, ?, ?, ?,
        ?, ?, ?, ?,
        ?, ?, ?,
        ?, ?, ?,
        ?, ?, ?,
        ?, ?, ?,
        ?, ?, ?,
        ?, ?
    )
"""

chunk_size   = 2000
total_chunks = (len(rows) + chunk_size - 1) // chunk_size
inserted     = 0

for i in range(0, len(rows), chunk_size):
    chunk     = rows[i:i + chunk_size]
    chunk_num = i // chunk_size + 1

    for row in chunk:
        session.sql(insert_sql, params=list(row)).collect()

    inserted += len(chunk)
    pct = round(chunk_num / total_chunks * 100)
    print(f"  [{pct:>3}%] Chunk {chunk_num}/{total_chunks} — {inserted:,}/{len(rows):,} rows")

# ─── STEP 4: VERIFY ───────────────────────────────────────────
print("\n📊 Verifying...")
result = session.sql("""
    SELECT COUNT(*)            AS total,
           MIN(snapshot_time)  AS from_time,
           MAX(snapshot_time)  AS to_time
    FROM LUMISENSE_DB.LUMISENSE_SCHEMA.fact_dashboard
""").collect()[0]

print(f"  Total rows : {result['TOTAL']:,}")
print(f"  From       : {result['FROM_TIME']}")
print(f"  To         : {result['TO_TIME']}")
print("\n🎉 fact_dashboard populated successfully!")